In [2]:
import requests
import pandas as pd

index_url = "https://gbfs.citibikenyc.com/gbfs/2.3/gbfs.json"
feeds = requests.get(index_url).json()["data"]["en"]["feeds"]
feed_urls = {feed["name"]: feed["url"] for feed in feeds}

station_info = requests.get(feed_urls["station_information"]).json()["data"]["stations"]
station_status = requests.get(feed_urls["station_status"]).json()["data"]["stations"]

stations = pd.DataFrame(station_info).merge(
    pd.DataFrame(station_status),
    on="station_id",
    how="left",
    suffixes=("_info", "_status"),
)

stations.head()

,station_id,name,short_name,lon,lat,region_id,capacity,rental_uris,is_charging,num_bikes_available,...,num_docks_available,num_docks_disabled,is_installed,is_renting,is_returning,last_reported,vehicle_types_available,num_ebikes_available,num_scooters_available,num_scooters_unavailable
0,3f05490b-a3d8-4d15-bd25-0212555b5dff,Fort Hamilton Pkwy & E 5 St,3240.15,-73.976730,40.649037,71,0,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,False,0,...,0,0,0,0,0,86400,"[{'vehicle_type_id': '1', 'count': 0}, {'vehic...",0,NaN,NaN
1,475c44e1-31e3-4c60-b071-af2ca3f7618c,Prospect Pl & Nostrand Ave,4025.02,-73.950300,40.674280,71,0,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,False,0,...,0,0,0,0,0,86400,"[{'vehicle_type_id': '1', 'count': 0}, {'vehic...",0,NaN,NaN
2,3eb10c63-ea4c-4161-b07d-a29034f7367a,Brook Ave & E 141 St,7740.08,-73.917875,40.809619,71,0,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,False,0,...,0,0,0,0,0,86400,"[{'vehicle_type_id': '1', 'count': 0}, {'vehic...",0,NaN,NaN
3,ba33c334-e17d-4478-ae3c-9c2f4dbfbb51,51 Ave & 61 St,5923.02,-73.903870,40.735680,71,0,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,False,0,...,0,0,0,0,0,86400,"[{'vehicle_type_id': '1', 'count': 0}, {'vehic...",0,NaN,NaN
4,06439006-11b6-44f0-8545-c9d39035f32a,Vesey St & Church St,5216.06,-74.010472,40.712220,71,78,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,False,0,...,0,0,0,0,0,86400,"[{'vehicle_type_id': '1', 'count': 0}, {'vehic...",0,NaN,NaN


In [4]:
import re
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from io import BytesIO
from zipfile import ZipFile

BASE_URL = "https://s3.amazonaws.com/tripdata"
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
START, END = "202401", "202609"

USECOLS = {
    "ride_id", "rideable_type", "started_at", "ended_at",
    "start_station_name", "start_station_id",
    "end_station_name", "end_station_id",
    "start_lat", "start_lng", "end_lat", "end_lng", "member_casual",
}


def list_keys() -> list[str]:
    """Return every key in the bucket (handles S3 pagination)."""
    keys, token = [], None
    while True:
        params = {"list-type": 2}
        if token:
            params["continuation-token"] = token
        r = requests.get(BASE_URL + "/", params=params, timeout=30)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        keys += [k.text for k in root.findall("s3:Contents/s3:Key", NS)]
        if root.findtext("s3:IsTruncated", namespaces=NS) == "true":
            token = root.findtext("s3:NextContinuationToken", namespaces=NS)
        else:
            return keys


def get_links() -> list[str]:
    # tolerates both "...-tripdata.zip" and "...-tripdata.csv.zip"
    pattern = re.compile(r"^(\d{6})-citibike-tripdata(\.csv)?\.zip$")
    return [
        f"{BASE_URL}/{k}"
        for k in sorted(list_keys())
        if (m := pattern.match(k)) and START <= m.group(1) <= END
    ]


def load_month(url: str) -> pd.DataFrame:
    response = requests.get(url, timeout=300)
    response.raise_for_status()

    with ZipFile(BytesIO(response.content)) as archive:
        csv_files = [
            n for n in archive.namelist()
            if n.endswith(".csv") and not n.startswith("__MACOSX")
        ]
        return pd.concat(
            (
                pd.read_csv(
                    archive.open(n),
                    usecols=lambda c: c in USECOLS,
                    parse_dates=["started_at", "ended_at"],
                    dtype={
                        "rideable_type": "category",
                        "member_casual": "category",
                        "start_station_id": "string",
                        "end_station_id": "string",
                    },
                )
                for n in csv_files
            ),
            ignore_index=True,
        )


links = get_links()
print(f"Found {len(links)} files")

frames = []
for url in links:
    try:
        df = load_month(url)
        frames.append(df)
        print(f"Loaded {url.rsplit('/', 1)[-1]} ({len(df):,} rows)")
    except (requests.RequestException, ValueError) as e:
        print(f"Skipped {url}: {e}")

rides = pd.concat(frames, ignore_index=True)
rides["year_month"] = rides["started_at"].dt.to_period("M")

print(f"\n{len(rides):,} total rides")
print(rides["year_month"].value_counts().sort_index())

Found 33 files
Loaded 202401-citibike-tripdata.zip (1,888,085 rows)
Loaded 202402-citibike-tripdata.zip (2,121,501 rows)
Loaded 202403-citibike-tripdata.zip (2,663,295 rows)


: 